In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')
        

/kaggle/input/datasets/kaushikar/drone-usat/DIAT-uSAT_dataset/Bird+mini-helicopter_1/Bird+2_Blade_rotor_1/figure370.jpg
/kaggle/input/datasets/kaushikar/drone-usat/DIAT-uSAT_dataset/Bird+mini-helicopter_1/Bird+2_Blade_rotor_1/figure190.jpg
/kaggle/input/datasets/kaushikar/drone-usat/DIAT-uSAT_dataset/Bird+mini-helicopter_1/Bird+2_Blade_rotor_1/figure175.jpg
/kaggle/input/datasets/kaushikar/drone-usat/DIAT-uSAT_dataset/Bird+mini-helicopter_1/Bird+2_Blade_rotor_1/figure287.jpg
/kaggle/input/datasets/kaushikar/drone-usat/DIAT-uSAT_dataset/Bird+mini-helicopter_1/Bird+2_Blade_rotor_1/figure135.jpg
/kaggle/input/datasets/kaushikar/drone-usat/DIAT-uSAT_dataset/Bird+mini-helicopter_1/Bird+2_Blade_rotor_1/figure121.jpg
/kaggle/input/datasets/kaushikar/drone-usat/DIAT-uSAT_dataset/Bird+mini-helicopter_1/Bird+2_Blade_rotor_1/figure149.jpg
/kaggle/input/datasets/kaushikar/drone-usat/DIAT-uSAT_dataset/Bird+mini-helicopter_1/Bird+2_Blade_rotor_1/figure265.jpg
/kaggle/input/datasets/kaushikar/drone-u

In [2]:
import os
import glob
from PIL import Image
from collections import defaultdict, Counter

ROOT = '/kaggle/input/datasets/kaushikar/drone-usat/DIAT-uSAT_dataset'

CLASS_MAP = {
    '3_long_blade_rotor':       '3_long_blade_rotor',
    '3_short_blade_rotor_1':    '3_short_blade_rotor',
    '3_short_blade_rotor_2':    '3_short_blade_rotor',
    'Bird':                     'Bird',
    'Bird+mini-helicopter_1':   'Bird+mini-helicopter',
    'Bird+mini-helicopter_2':   'Bird+mini-helicopter',
    'RC plane_1':               'RC_plane',
    'RC plane_2':               'RC_plane',
    'drone_1':                  'drone',
    'drone_2':                  'drone',
}


def list_images(folder):
    exts = ('*.png', '*.jpg', '*.jpeg', '*.bmp', '*.PNG', '*.JPG', '*.JPEG')
    files = []
    for e in exts:
        files += glob.glob(os.path.join(folder, '**', e), recursive=True)
    return files


per_class_sizes = defaultdict(Counter)
per_class_modes = defaultdict(Counter)
per_class_fmts = defaultdict(Counter)
global_sizes = Counter()
total = 0

for top in sorted(os.listdir(ROOT)):
    if top not in CLASS_MAP:
        continue
    cls = CLASS_MAP[top]
    for f in list_images(os.path.join(ROOT, top)):
        try:
            with Image.open(f) as im:
                per_class_sizes[cls][im.size] += 1
                per_class_modes[cls][im.mode] += 1
                per_class_fmts[cls][im.format] += 1
                global_sizes[im.size] += 1
                total += 1
        except Exception as e:
            print('UNREADABLE:', f, e)

print(f'Total readable images: {total}\n')
print('Per merged class:')
for cls in sorted(per_class_sizes):
    sizes = per_class_sizes[cls]
    modes = per_class_modes[cls]
    fmts = per_class_fmts[cls]
    n = sum(sizes.values())
    print(f'\n  {cls}  (n={n})')
    print(f'    sizes (W x H): {dict(sizes)}')
    print(f'    modes        : {dict(modes)}')
    print(f'    formats      : {dict(fmts)}')

print('\n' + '=' * 60)
print('Global size distribution (W x H):')
for sz, c in global_sizes.most_common():
    print(f'  {sz}: {c}')

print('\nVERDICT:')
print(f'  Distinct sizes across dataset : {len(global_sizes)}')
if len(global_sizes) == 1:
    print('  -> Uniform dimensions. Cadence sampling is consistent across all classes.')
else:
    print('  -> MIXED dimensions. Native time-resolution differs; normalize width before CVD.')

Total readable images: 4849

Per merged class:

  3_long_blade_rotor  (n=799)
    sizes (W x H): {(1400, 1050): 799}
    modes        : {'RGB': 799}
    formats      : {'JPEG': 799}

  3_short_blade_rotor  (n=800)
    sizes (W x H): {(1400, 1050): 800}
    modes        : {'RGB': 800}
    formats      : {'JPEG': 800}

  Bird  (n=800)
    sizes (W x H): {(700, 525): 800}
    modes        : {'RGB': 800}
    formats      : {'JPEG': 800}

  Bird+mini-helicopter  (n=815)
    sizes (W x H): {(1400, 1050): 815}
    modes        : {'RGB': 815}
    formats      : {'JPEG': 815}

  RC_plane  (n=800)
    sizes (W x H): {(1400, 1050): 800}
    modes        : {'RGB': 800}
    formats      : {'JPEG': 800}

  drone  (n=835)
    sizes (W x H): {(1400, 1050): 835}
    modes        : {'RGB': 835}
    formats      : {'JPEG': 835}

Global size distribution (W x H):
  (1400, 1050): 4049
  (700, 525): 800

VERDICT:
  Distinct sizes across dataset : 2
  -> MIXED dimensions. Native time-resolution differs; norm

In [3]:
import os, glob, random, time, warnings
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from scipy import stats
import subprocess

warnings.filterwarnings('ignore')

try:
    from thop import profile as thop_profile
except Exception:
    subprocess.run(['pip', 'install', '-q', 'thop'])
    from thop import profile as thop_profile

SEEDS       = [42, 7, 13, 99, 2025]
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ROOT        = '/kaggle/input/datasets/kaushikar/drone-usat/DIAT-uSAT_dataset'
WORK        = '/kaggle/working'
IMG_H       = 176
IMG_W       = 512
BATCH       = 64
EPOCHS      = 60
LR          = 1e-3
N_CLASSES   = 6

CLASS_MAP = {
    '3_long_blade_rotor':     '3_long_blade_rotor',
    '3_short_blade_rotor_1':  '3_short_blade_rotor',
    '3_short_blade_rotor_2':  '3_short_blade_rotor',
    'Bird':                   'Bird',
    'Bird+mini-helicopter_1': 'Bird+mini-helicopter',
    'Bird+mini-helicopter_2': 'Bird+mini-helicopter',
    'RC plane_1':             'RC_plane',
    'RC plane_2':             'RC_plane',
    'drone_1':                'drone',
    'drone_2':                'drone',
}
CLASSES     = sorted(set(CLASS_MAP.values()))
CLS2IDX     = {c: i for i, c in enumerate(CLASSES)}
print('Device:', DEVICE)
print('Classes:', CLASSES)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 102.6 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

Device: cuda
Classes: ['3_long_blade_rotor', '3_short_blade_rotor', 'Bird', 'Bird+mini-helicopter', 'RC_plane', 'drone']


In [4]:
def list_images(folder):
    exts = ('*.png','*.jpg','*.jpeg','*.PNG','*.JPG','*.JPEG')
    out  = []
    for e in exts:
        out += glob.glob(os.path.join(folder,'**',e), recursive=True)
    return out

def load_and_preprocess(path):
    img = Image.open(path).convert('L')
    arr = np.asarray(img, dtype=np.float32)
    mask = arr < 240
    if mask.any():
        rows = np.where(mask.any(axis=1))[0]
        cols = np.where(mask.any(axis=0))[0]
        arr  = arr[rows[0]:rows[-1]+1, cols[0]:cols[-1]+1]
    out = np.asarray(
        Image.fromarray(arr.astype(np.uint8)).resize((IMG_W, IMG_H), Image.BILINEAR),
        dtype=np.uint8)
    return out

paths, labels = [], []
for top in sorted(os.listdir(ROOT)):
    if top in CLASS_MAP:
        c = CLS2IDX[CLASS_MAP[top]]
        for f in list_images(os.path.join(ROOT, top)):
            paths.append(f)
            labels.append(c)
labels = np.array(labels, dtype=np.int64)

print('Loading images...')
data = np.zeros((len(paths), IMG_H, IMG_W), dtype=np.uint8)
for i, p in enumerate(paths):
    data[i] = load_and_preprocess(p)
print(f'Loaded: {data.shape}')

for c in CLASSES:
    print(f'  {c}: {(labels==CLS2IDX[c]).sum()}')

idx = np.arange(len(paths))
tr_idx, tmp_idx = train_test_split(idx, test_size=0.2, stratify=labels, random_state=42)
va_idx, te_idx  = train_test_split(tmp_idx, test_size=0.5, stratify=labels[tmp_idx], random_state=42)
print(f'Train/Val/Test: {len(tr_idx)}/{len(va_idx)}/{len(te_idx)}')

np.save(os.path.join(WORK,'data.npy'),   data)
np.save(os.path.join(WORK,'labels.npy'), labels)
np.save(os.path.join(WORK,'tr_idx.npy'), tr_idx)
np.save(os.path.join(WORK,'va_idx.npy'), va_idx)
np.save(os.path.join(WORK,'te_idx.npy'), te_idx)
print('Split saved. No model has seen the test set.')

Loading images...
Loaded: (4849, 176, 512)
  3_long_blade_rotor: 799
  3_short_blade_rotor: 800
  Bird: 800
  Bird+mini-helicopter: 815
  RC_plane: 800
  drone: 835
Train/Val/Test: 3879/485/485
Split saved. No model has seen the test set.


In [5]:
class MDDataset(Dataset):
    def __init__(self, data, labels, indices, train=False):
        self.data    = data
        self.labels  = labels
        self.indices = indices
        self.train   = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        j   = self.indices[i]
        img = self.data[j].astype(np.float32) / 255.0
        if self.train:
            if random.random() < 0.5:
                img = img[:, ::-1].copy()
            if random.random() < 0.5:
                img = np.clip(img * random.uniform(0.9,1.1), 0.0, 1.0)
            if random.random() < 0.3:
                f0  = random.randint(0, IMG_H-20)
                img[f0:f0+random.randint(5,20), :] = 0.0
            if random.random() < 0.3:
                t0  = random.randint(0, IMG_W-25)
                img[:, t0:t0+random.randint(5,25)] = 0.0
        return torch.from_numpy(img).unsqueeze(0), int(self.labels[j])

In [6]:
class ConvBNSiLU(nn.Module):
    def __init__(self, ci, co, k=3, s=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(ci, co, k, s, k//2, bias=False),
            nn.BatchNorm2d(co),
            nn.SiLU(inplace=True))
    def forward(self, x): return self.block(x)

class ConvBNReLU(nn.Module):
    def __init__(self, ci, co, k=3, s=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(ci, co, k, s, k//2, bias=False),
            nn.BatchNorm2d(co),
            nn.ReLU(inplace=True))
    def forward(self, x): return self.block(x)

class DSConv2d(nn.Module):
    def __init__(self, ci, co, s=1):
        super().__init__()
        self.dw  = nn.Conv2d(ci, ci, 3, s, 1, groups=ci, bias=False)
        self.pw  = nn.Conv2d(ci, co, 1, bias=False)
        self.bn  = nn.BatchNorm2d(co)
        self.act = nn.SiLU(inplace=True)
    def forward(self, x): return self.act(self.bn(self.pw(self.dw(x))))

class Conv1dBNSiLU(nn.Module):
    def __init__(self, ci, co, k=5, s=2):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv1d(ci, co, k, s, k//2, bias=False),
            nn.BatchNorm1d(co),
            nn.SiLU(inplace=True))
    def forward(self, x): return self.block(x)

class ResBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.b1  = ConvBNSiLU(ch, ch//2, 1)
        self.b2  = ConvBNSiLU(ch//2, ch, 3)
    def forward(self, x): return x + self.b2(self.b1(x))

class SEBlock(nn.Module):
    def __init__(self, ch, r=4):
        super().__init__()
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(ch, ch//r, bias=False),
            nn.SiLU(inplace=True),
            nn.Linear(ch//r, ch, bias=False),
            nn.Sigmoid())
    def forward(self, x):
        return x * self.fc(x).view(x.size(0), -1, 1, 1)

class CVDLayer(nn.Module):
    """
    In-network CVD: rfft along the time axis of the 2D feature map.
    Input : (B, C, H, W)  — C channels, H=frequency bins, W=time bins
    Output: (B, C, H, W//2+1)  magnitude spectrum, log-compressed
    No learnable parameters — parameter-free physics operator.
    """
    def forward(self, x):
        cvd = torch.fft.rfft(x, dim=3)
        mag = cvd.abs()
        return torch.log1p(mag)

In [7]:
def make_head(in_dim, n_classes, dropout=0.3):
    return nn.Sequential(
        nn.Linear(in_dim, max(in_dim//2, 32)),
        nn.BatchNorm1d(max(in_dim//2, 32)),
        nn.SiLU(inplace=True),
        nn.Dropout(dropout),
        nn.Linear(max(in_dim//2, 32), n_classes))

def make_stem(width):
    return nn.Sequential(
        ConvBNSiLU(1, 16, 3, 2),
        ConvBNSiLU(16, width, 3, 2))

def make_velocity_branch(width, out_ch):
    return nn.Sequential(
        Conv1dBNSiLU(width, out_ch, 5, 2),
        Conv1dBNSiLU(out_ch, out_ch, 5, 2))

def make_cadence_branch(width, out_ch):
    return nn.Sequential(
        Conv1dBNSiLU(width, out_ch, 5, 2),
        Conv1dBNSiLU(out_ch, out_ch, 5, 2))

def make_joint_branch(width, out_ch):
    return nn.Sequential(
        DSConv2d(width, out_ch, s=2),
        DSConv2d(out_ch, out_ch, s=2))


class VelocityOnly(nn.Module):
    """Ablation: Doppler marginal spectrum only (proven winner from round 1)."""
    def __init__(self, n=N_CLASSES, w=32, oc=64):
        super().__init__()
        self.stem     = make_stem(w)
        self.vel      = make_velocity_branch(w, oc)
        self.gap      = nn.AdaptiveAvgPool1d(1)
        self.head     = make_head(oc, n)
    def forward(self, x):
        f = self.stem(x)
        v = self.gap(self.vel(f.mean(dim=3))).flatten(1)
        return self.head(v)


class NaiveTemporalOnly(nn.Module):
    """Ablation: raw mean-over-frequency (the failed old temporal branch)."""
    def __init__(self, n=N_CLASSES, w=32, oc=64):
        super().__init__()
        self.stem = make_stem(w)
        self.tmp  = make_cadence_branch(w, oc)
        self.gap  = nn.AdaptiveAvgPool1d(1)
        self.head = make_head(oc, n)
    def forward(self, x):
        f = self.stem(x)
        t = self.gap(self.tmp(f.mean(dim=2))).flatten(1)
        return self.head(t)


class CVDOnly(nn.Module):
    """Ablation: in-network CVD (cadence) stream only."""
    def __init__(self, n=N_CLASSES, w=32, oc=64):
        super().__init__()
        self.stem = make_stem(w)
        self.cvd  = CVDLayer()
        self.cad  = nn.Sequential(
            DSConv2d(w, oc, s=2),
            DSConv2d(oc, oc, s=2))
        self.gap  = nn.AdaptiveAvgPool2d(1)
        self.head = make_head(oc, n)
    def forward(self, x):
        f = self.stem(x)
        c = self.gap(self.cad(self.cvd(f))).flatten(1)
        return self.head(c)


class VelNaiveTemporal(nn.Module):
    """Ablation: velocity + naive temporal (round 1 full model equivalent)."""
    def __init__(self, n=N_CLASSES, w=32, oc=64):
        super().__init__()
        self.stem = make_stem(w)
        self.vel  = make_velocity_branch(w, oc)
        self.tmp  = make_cadence_branch(w, oc)
        self.gap  = nn.AdaptiveAvgPool1d(1)
        self.head = make_head(oc*2, n)
    def forward(self, x):
        f = self.stem(x)
        v = self.gap(self.vel(f.mean(dim=3))).flatten(1)
        t = self.gap(self.tmp(f.mean(dim=2))).flatten(1)
        return self.head(torch.cat([v,t],1))


class VelCVD_Concat(nn.Module):
    """Proposed: velocity + CVD, plain concatenation fusion."""
    def __init__(self, n=N_CLASSES, w=32, oc=64):
        super().__init__()
        self.stem = make_stem(w)
        self.vel  = make_velocity_branch(w, oc)
        self.cvd  = CVDLayer()
        self.cad  = nn.Sequential(
            DSConv2d(w, oc, s=2),
            DSConv2d(oc, oc, s=2))
        self.gap1 = nn.AdaptiveAvgPool1d(1)
        self.gap2 = nn.AdaptiveAvgPool2d(1)
        self.head = make_head(oc*2, n)
    def forward(self, x):
        f = self.stem(x)
        v = self.gap1(self.vel(f.mean(dim=3))).flatten(1)
        c = self.gap2(self.cad(self.cvd(f))).flatten(1)
        return self.head(torch.cat([v,c],1))


class VelCVD_Gate(nn.Module):
    """Proposed: velocity + CVD, learned gated fusion."""
    def __init__(self, n=N_CLASSES, w=32, oc=64):
        super().__init__()
        self.stem  = make_stem(w)
        self.vel   = make_velocity_branch(w, oc)
        self.cvd   = CVDLayer()
        self.cad   = nn.Sequential(
            DSConv2d(w, oc, s=2),
            DSConv2d(oc, oc, s=2))
        self.gap1  = nn.AdaptiveAvgPool1d(1)
        self.gap2  = nn.AdaptiveAvgPool2d(1)
        self.gate  = nn.Sequential(
            nn.Linear(oc*2, oc*2),
            nn.Sigmoid())
        self.head  = make_head(oc*2, n)
    def forward(self, x):
        f   = self.stem(x)
        v   = self.gap1(self.vel(f.mean(dim=3))).flatten(1)
        c   = self.gap2(self.cad(self.cvd(f))).flatten(1)
        cat = torch.cat([v,c],1)
        return self.head(self.gate(cat) * cat)


class VelCVD_Joint(nn.Module):
    """Proposed: velocity + CVD + joint 2D branch (three streams)."""
    def __init__(self, n=N_CLASSES, w=32, oc=64):
        super().__init__()
        self.stem  = make_stem(w)
        self.vel   = make_velocity_branch(w, oc)
        self.cvd   = CVDLayer()
        self.cad   = nn.Sequential(
            DSConv2d(w, oc, s=2),
            DSConv2d(oc, oc, s=2))
        self.jnt   = make_joint_branch(w, oc)
        self.gap1  = nn.AdaptiveAvgPool1d(1)
        self.gap2  = nn.AdaptiveAvgPool2d(1)
        self.head  = make_head(oc*3, n)
    def forward(self, x):
        f = self.stem(x)
        v = self.gap1(self.vel(f.mean(dim=3))).flatten(1)
        c = self.gap2(self.cad(self.cvd(f))).flatten(1)
        j = self.gap2(self.jnt(f)).flatten(1)
        return self.head(torch.cat([v,c,j],1))


class VelCVD_Res(nn.Module):
    """Layer experiment: velocity + CVD with residual blocks in each branch."""
    def __init__(self, n=N_CLASSES, w=32, oc=64):
        super().__init__()
        self.stem  = make_stem(w)
        self.vel   = nn.Sequential(
            Conv1dBNSiLU(w, oc, 5, 2),
            Conv1dBNSiLU(oc, oc, 5, 1),
            Conv1dBNSiLU(oc, oc, 5, 2))
        self.cvd   = CVDLayer()
        self.cad   = nn.Sequential(
            DSConv2d(w, oc, s=2),
            ResBlock(oc) if False else DSConv2d(oc, oc, s=1),
            DSConv2d(oc, oc, s=2))
        self.gap1  = nn.AdaptiveAvgPool1d(1)
        self.gap2  = nn.AdaptiveAvgPool2d(1)
        self.head  = make_head(oc*2, n)
    def forward(self, x):
        f = self.stem(x)
        v = self.gap1(self.vel(f.mean(dim=3))).flatten(1)
        c = self.gap2(self.cad(self.cvd(f))).flatten(1)
        return self.head(torch.cat([v,c],1))


class VelCVD_Deep(nn.Module):
    """Layer experiment: deeper stem (3 stages) + velocity + CVD."""
    def __init__(self, n=N_CLASSES, w=32, oc=64):
        super().__init__()
        self.stem  = nn.Sequential(
            ConvBNSiLU(1, 16, 3, 2),
            ConvBNSiLU(16, w, 3, 2),
            ConvBNSiLU(w, w, 3, 1))
        self.vel   = nn.Sequential(
            Conv1dBNSiLU(w, oc, 5, 2),
            Conv1dBNSiLU(oc, oc, 5, 2),
            Conv1dBNSiLU(oc, oc, 3, 1))
        self.cvd   = CVDLayer()
        self.cad   = nn.Sequential(
            DSConv2d(w, oc, s=2),
            DSConv2d(oc, oc, s=2),
            DSConv2d(oc, oc, s=1))
        self.gap1  = nn.AdaptiveAvgPool1d(1)
        self.gap2  = nn.AdaptiveAvgPool2d(1)
        self.head  = make_head(oc*2, n)
    def forward(self, x):
        f = self.stem(x)
        v = self.gap1(self.vel(f.mean(dim=3))).flatten(1)
        c = self.gap2(self.cad(self.cvd(f))).flatten(1)
        return self.head(torch.cat([v,c],1))


class VelCVD_SE(nn.Module):
    """Layer experiment: velocity + CVD + SE attention on stem features."""
    def __init__(self, n=N_CLASSES, w=32, oc=64):
        super().__init__()
        self.stem  = make_stem(w)
        self.se    = SEBlock(w)
        self.vel   = make_velocity_branch(w, oc)
        self.cvd   = CVDLayer()
        self.cad   = nn.Sequential(
            DSConv2d(w, oc, s=2),
            DSConv2d(oc, oc, s=2))
        self.gap1  = nn.AdaptiveAvgPool1d(1)
        self.gap2  = nn.AdaptiveAvgPool2d(1)
        self.head  = make_head(oc*2, n)
    def forward(self, x):
        f = self.se(self.stem(x))
        v = self.gap1(self.vel(f.mean(dim=3))).flatten(1)
        c = self.gap2(self.cad(self.cvd(f))).flatten(1)
        return self.head(torch.cat([v,c],1))


class VelCVD_Wide(nn.Module):
    """Width experiment: w=48, oc=96."""
    def __init__(self, n=N_CLASSES, w=48, oc=96):
        super().__init__()
        self.stem  = make_stem(w)
        self.vel   = make_velocity_branch(w, oc)
        self.cvd   = CVDLayer()
        self.cad   = nn.Sequential(
            DSConv2d(w, oc, s=2),
            DSConv2d(oc, oc, s=2))
        self.gap1  = nn.AdaptiveAvgPool1d(1)
        self.gap2  = nn.AdaptiveAvgPool2d(1)
        self.head  = make_head(oc*2, n)
    def forward(self, x):
        f = self.stem(x)
        v = self.gap1(self.vel(f.mean(dim=3))).flatten(1)
        c = self.gap2(self.cad(self.cvd(f))).flatten(1)
        return self.head(torch.cat([v,c],1))


class VelCVD_Narrow(nn.Module):
    """Width experiment: w=16, oc=32 — minimum capacity."""
    def __init__(self, n=N_CLASSES, w=16, oc=32):
        super().__init__()
        self.stem  = make_stem(w)
        self.vel   = make_velocity_branch(w, oc)
        self.cvd   = CVDLayer()
        self.cad   = nn.Sequential(
            DSConv2d(w, oc, s=2),
            DSConv2d(oc, oc, s=2))
        self.gap1  = nn.AdaptiveAvgPool1d(1)
        self.gap2  = nn.AdaptiveAvgPool2d(1)
        self.head  = make_head(oc*2, n)
    def forward(self, x):
        f = self.stem(x)
        v = self.gap1(self.vel(f.mean(dim=3))).flatten(1)
        c = self.gap2(self.cad(self.cvd(f))).flatten(1)
        return self.head(torch.cat([v,c],1))

In [8]:
data   = np.load(os.path.join(WORK,'data.npy'))
labels = np.load(os.path.join(WORK,'labels.npy'))
tr_idx = np.load(os.path.join(WORK,'tr_idx.npy'))
va_idx = np.load(os.path.join(WORK,'va_idx.npy'))
te_idx = np.load(os.path.join(WORK,'te_idx.npy'))

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def train_one_seed(model_fn, seed):
    set_seed(seed)
    model = model_fn().to(DEVICE)
    opt   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-5)
    crit  = nn.CrossEntropyLoss(label_smoothing=0.05)
    scl   = torch.amp.GradScaler('cuda', enabled=DEVICE.type=='cuda')

    tl = DataLoader(MDDataset(data,labels,tr_idx,True),
                    batch_size=BATCH, shuffle=True,
                    num_workers=2, pin_memory=True, drop_last=True)
    vl = DataLoader(MDDataset(data,labels,va_idx,False),
                    batch_size=BATCH, shuffle=False,
                    num_workers=2, pin_memory=True)

    best_acc, best_state = 0.0, None
    for ep in range(EPOCHS):
        model.train()
        for x,y in tl:
            x,y = x.to(DEVICE,non_blocking=True), y.to(DEVICE,non_blocking=True)
            opt.zero_grad()
            with torch.amp.autocast('cuda', enabled=DEVICE.type=='cuda'):
                loss = crit(model(x), y)
            scl.scale(loss).backward()
            scl.step(opt); scl.update()
        sched.step()
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for x,y in vl:
                x,y = x.to(DEVICE),y.to(DEVICE)
                correct += (model(x).argmax(1)==y).sum().item()
                total   += y.size(0)
        va = correct/total
        if va > best_acc:
            best_acc   = va
            best_state = {k:v.detach().cpu().clone()
                          for k,v in model.state_dict().items()}

    model.load_state_dict(best_state)
    model.eval()

    tel  = DataLoader(MDDataset(data,labels,te_idx,False),
                      batch_size=BATCH, shuffle=False,
                      num_workers=2, pin_memory=True)
    yt, yp = [], []
    with torch.no_grad():
        for x,y in tel:
            yp.append(model(x.to(DEVICE)).argmax(1).cpu().numpy())
            yt.append(y.numpy())
    yt = np.concatenate(yt); yp = np.concatenate(yp)

    n_params = sum(p.numel() for p in model.parameters())
    size_mb  = sum(v.numel()*v.element_size()
                   for v in model.state_dict().values())/1e6
    dummy    = torch.randn(1,1,IMG_H,IMG_W).to(DEVICE)
    try:
        macs,_ = thop_profile(model, inputs=(dummy,), verbose=False)
        flops  = 2*macs/1e9
    except Exception:
        flops  = float('nan')

    return accuracy_score(yt,yp), yt, yp, n_params, size_mb, flops

def mcnemar_test(yp_a, yp_ref, yt):
    correct_a   = (yp_a   == yt)
    correct_ref = (yp_ref == yt)
    b = ((~correct_a) & correct_ref).sum()
    c = (correct_a   & (~correct_ref)).sum()
    if b+c == 0:
        return 1.0
    chi2 = (abs(b-c)-1)**2 / (b+c)
    return float(1 - stats.chi2.cdf(chi2, df=1))

def run_variant(name, model_fn):
    accs, all_yp, all_yt = [], [], []
    params, size, flops  = None, None, None
    for s in SEEDS:
        acc, yt, yp, p, sz, fl = train_one_seed(model_fn, s)
        accs.append(acc); all_yp.append(yp); all_yt.append(yt)
        if params is None: params,size,flops = p,sz,fl

    accs   = np.array(accs)
    mean   = accs.mean()
    std    = accs.std(ddof=1)
    ci95   = stats.t.interval(0.95, df=len(accs)-1,
                               loc=mean, scale=stats.sem(accs))
    return dict(name=name, accs=accs, mean=mean, std=std,
                ci95=ci95, all_yp=all_yp, all_yt=all_yt,
                params=params, size=size, flops=flops)

In [9]:
VARIANTS = [
    ('1. Velocity only (baseline)',          lambda: VelocityOnly()),
    ('2. Naive temporal only',               lambda: NaiveTemporalOnly()),
    ('3. CVD only',                          lambda: CVDOnly()),
    ('4. Vel + naive temporal',              lambda: VelNaiveTemporal()),
    ('5. Vel + CVD concat (proposed)',       lambda: VelCVD_Concat()),
    ('6. Vel + CVD gated fusion',            lambda: VelCVD_Gate()),
    ('7. Vel + CVD + joint 2D',              lambda: VelCVD_Joint()),
    ('8. Vel + CVD deeper branches',         lambda: VelCVD_Deep()),
    ('9. Vel + CVD + SE attention',          lambda: VelCVD_SE()),
    ('10. Vel + CVD wide (w=48)',            lambda: VelCVD_Wide()),
    ('11. Vel + CVD narrow (w=16)',          lambda: VelCVD_Narrow()),
]

print('='*80)
print('MULTI-SEED ABLATION  (5 seeds × 11 variants = 55 training runs)')
print('='*80)

results = {}
for name, fn in VARIANTS:
    print(f'\nRunning: {name}')
    r = run_variant(name, fn)
    results[name] = r
    print(f'  Seeds: {[f"{a*100:.2f}" for a in r["accs"]]}')
    print(f'  Mean±Std: {r["mean"]*100:.2f}% ± {r["std"]*100:.2f}%')
    print(f'  95% CI:   [{r["ci95"][0]*100:.2f}%, {r["ci95"][1]*100:.2f}%]')
    print(f'  Params: {r["params"]:,}  Size: {r["size"]:.3f}MB  FLOPs: {r["flops"]:.4f}G')

MULTI-SEED ABLATION  (5 seeds × 11 variants = 55 training runs)

Running: 1. Velocity only (baseline)
  Seeds: ['99.38', '99.59', '99.18', '99.59', '99.38']
  Mean±Std: 99.42% ± 0.17%
  95% CI:   [99.21%, 99.64%]
  Params: 38,166  Size: 0.154MB  FLOPs: 0.0636G

Running: 2. Naive temporal only
  Seeds: ['97.73', '96.29', '98.14', '97.94', '97.32']
  Mean±Std: 97.48% ± 0.73%
  95% CI:   [96.57%, 98.40%]
  Params: 38,166  Size: 0.154MB  FLOPs: 0.0654G

Running: 3. CVD only
  Seeds: ['96.91', '97.94', '97.53', '97.73', '98.97']
  Mean±Std: 97.81% ± 0.75%
  95% CI:   [96.88%, 98.75%]
  Params: 14,454  Size: 0.060MB  FLOPs: 0.0684G

Running: 4. Vel + naive temporal
  Seeds: ['99.79', '99.59', '99.59', '99.59', '100.00']
  Mean±Std: 99.71% ± 0.18%
  95% CI:   [99.48%, 99.94%]
  Params: 75,574  Size: 0.305MB  FLOPs: 0.0663G

Running: 5. Vel + CVD concat (proposed)
  Seeds: ['99.18', '99.79', '99.79', '100.00', '99.38']
  Mean±Std: 99.63% ± 0.34%
  95% CI:   [99.21%, 100.05%]
  Params: 51,862  

In [10]:
ref_name = '1. Velocity only (baseline)'
ref      = results[ref_name]
ref_yp   = ref['all_yp'][0]
ref_yt   = ref['all_yt'][0]

print()
print('='*90)
print('ABLATION SUMMARY — McNemar p-values vs velocity-only baseline (seed 42)')
print('='*90)
header = f"{'Variant':<40} {'Mean%':>7} {'±Std':>6} {'CI95_lo':>8} {'CI95_hi':>8} {'Params':>9} {'MB':>6} {'FLOPs':>8} {'McNemar_p':>10}"
print(header)
print('-'*90)

for name, r in results.items():
    p_val = mcnemar_test(r['all_yp'][0], ref_yp, ref_yt)
    star  = '*' if p_val < 0.05 else ''
    print(f"{name:<40} "
          f"{r['mean']*100:>7.2f} "
          f"{r['std']*100:>6.2f} "
          f"{r['ci95'][0]*100:>8.2f} "
          f"{r['ci95'][1]*100:>8.2f} "
          f"{r['params']:>9,} "
          f"{r['size']:>6.3f} "
          f"{r['flops']:>8.4f} "
          f"{p_val:>10.4f}{star}")

print()
print('* = significant at p<0.05 vs velocity-only baseline')
print()
print('Per-seed breakdown:')
for name, r in results.items():
    seeds_str = '  '.join([f's{i+1}:{a*100:.2f}%' for i,a in enumerate(r['accs'])])
    print(f"  {name:<40} {seeds_str}")

print()
print('DECISION RULE:')
print('  Selected model = highest mean within CI overlapping the best mean,')
print('  with minimum parameters (SqueezeNet-style selection).')
print()

best_mean  = max(r['mean'] for r in results.values())
candidates = [(n,r) for n,r in results.items()
              if r['ci95'][0] <= best_mean <= r['ci95'][1] or
                 r['mean'] >= best_mean - r['std']]
candidates.sort(key=lambda x: x[1]['params'])
winner_name, winner = candidates[0]
print(f'  Winner: {winner_name}')
print(f'  Mean: {winner["mean"]*100:.2f}%  Params: {winner["params"]:,}  '
      f'Size: {winner["size"]:.3f}MB  FLOPs: {winner["flops"]:.4f}G')


ABLATION SUMMARY — McNemar p-values vs velocity-only baseline (seed 42)
Variant                                    Mean%   ±Std  CI95_lo  CI95_hi    Params     MB    FLOPs  McNemar_p
------------------------------------------------------------------------------------------
1. Velocity only (baseline)                99.42   0.17    99.21    99.64    38,166  0.154   0.0636     1.0000
2. Naive temporal only                     97.48   0.73    96.57    98.40    38,166  0.154   0.0654     0.0433*
3. CVD only                                97.81   0.75    96.88    98.75    14,454  0.060   0.0684     0.0060*
4. Vel + naive temporal                    99.71   0.18    99.48    99.94    75,574  0.305   0.0663     0.6171
5. Vel + CVD concat (proposed)             99.63   0.34    99.21   100.05    51,862  0.210   0.0693     1.0000
6. Vel + CVD gated fusion                  99.71   0.35    99.28   100.14    68,374  0.276   0.0693     1.0000
7. Vel + CVD + joint 2D                    99.42   0.27  